In [2]:
!pip install -q transformers easyocr pymupdf python-docx pillow opencv-python numpy

import os, re, fitz, torch, easyocr, numpy as np
from PIL import Image
from collections import Counter, defaultdict
from google.colab import drive
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.text import WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.enum.style import WD_STYLE_TYPE

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_model2/checkpoint-472"
INPUT_PDF = "/content/drive/MyDrive/GOST_Project/Денисова_ВКР.pdf"
WORK_DIR = "/content/drive/MyDrive/GOST_Project/gost_smart_work"
OUTPUT_DOCX = "/content/drive/MyDrive/GOST_Project/smart_result.docx"
IMAGES_DIR = os.path.join(WORK_DIR, "pages")
os.makedirs(IMAGES_DIR, exist_ok=True)

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = {0: 'BODY', 1: 'FIGURE', 2: 'FIGURE_CAPTION', 3: 'FOOTNOTE', 4: 'FORMULA',
            5: 'OTHER', 6: 'REFERENCE', 7: 'SECTION_HEADER', 8: 'TABLE',
            9: 'TABLE_CAPTION', 10: 'TITLE', 11: 'TOC'}

# PDF -> PNG
pdf = fitz.open(INPUT_PDF)
for page_num in range(len(pdf)):
    pix = pdf[page_num].get_pixmap(matrix=fitz.Matrix(2, 2))
    pix.save(os.path.join(IMAGES_DIR, f"page_{page_num+1}.png"))
print("PDF -> PNG")

reader = easyocr.Reader(['ru', 'en'], gpu=torch.cuda.is_available())

# ocr, предсказания, построение строк
def run_ocr(image_path):
    result = reader.readtext(image_path, paragraph=False)
    words, boxes = [], []
    for item in result:
        bbox, text = item[0], item[1].strip()
        if not text: continue
        xs, ys = [p[0] for p in bbox], [p[1] for p in bbox]
        words.append(text)
        boxes.append([int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))])
    return words, boxes

def normalize_box(box, width, height):
    return [int(1000 * box[0] / width), int(1000 * box[1] / height),
            int(1000 * box[2] / width), int(1000 * box[3] / height)]

def predict_page(image_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    words, boxes = run_ocr(image_path)
    if not words: return []
    norm_boxes = [normalize_box(b, width, height) for b in boxes]
    encoding = processor(image, words, boxes=norm_boxes, truncation=True,
                         padding="max_length", max_length=512, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**encoding)
    preds = outputs.logits.argmax(-1)[0]
    word_ids = encoding.word_ids()
    result, used = [], set()
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx in used: continue
        used.add(word_idx)
        result.append({"text": words[word_idx],
                       "label": id2label[preds[token_idx].item()],
                       "box": boxes[word_idx]})
    return result

def build_lines(predictions, y_threshold=18):
    if not predictions: return []
    predictions = sorted(predictions, key=lambda x: (x["box"][1], x["box"][0]))
    lines, current_line, prev_y = [], [], None
    for item in predictions:
        y_top = item["box"][1]
        if prev_y is None or abs(y_top - prev_y) > y_threshold:
            if current_line: lines.append(current_line)
            current_line = [item]
        else:
            current_line.append(item)
        prev_y = y_top
    if current_line: lines.append(current_line)

    result = []
    for line in lines:
        line = sorted(line, key=lambda x: x["box"][0])
        labels = [item["label"] for item in line]
        main_label = Counter(labels).most_common(1)[0][0]
        result.append({
            "text": " ".join(item["text"] for item in line).strip(),
            "label": main_label,
            "box": [min(item["box"][0] for item in line),
                    min(item["box"][1] for item in line),
                    max(item["box"][2] for item in line),
                    max(item["box"][3] for item in line)]
        })
    return result

print("Извлекаем кандидатов из модели.")
all_toc_candidates, all_ref_candidates = [], []
pages_data = {}

for image_name in sorted(os.listdir(IMAGES_DIR)):
    page_num = int(image_name.replace("page_", "").replace(".png", ""))
    image_path = os.path.join(IMAGES_DIR, image_name)
    lines = build_lines(predict_page(image_path))
    pages_data[page_num] = lines
    for line in lines:
        if line["label"] == "TOC":
            all_toc_candidates.append({**line, "page": page_num})
        elif line["label"] == "REFERENCE":
            all_ref_candidates.append({**line, "page": page_num})

print(f"Кандидатов TOC: {len(all_toc_candidates)}")
print(f"Кандидатов REFERENCE: {len(all_ref_candidates)}")

# фильтруем TOC
def is_toc_heading(text):
    return text.strip().lower() in ["содержание", "оглавление"]

def is_likely_toc_entry(text):
    text = text.strip()
    has_section_num = re.match(r'^\s*(\d+(\.\d+)*\.?\s+|\w+\.\s+)', text) is not None
    has_page_num = re.search(r'\d+\s*$', text) is not None
    if has_section_num and has_page_num:
        return True
    if has_section_num and len(text) < 80:
        return True
    return False

def filter_toc(candidates, pages_data):

    result = []

    toc_page = None

    for c in candidates:

        if is_toc_heading(c["text"]):

            toc_page = c["page"]

            break

    if toc_page is None:

        toc_page = 2

    page = toc_page

    while page in pages_data:

        added = 0

        for line in pages_data[page]:

            txt = line["text"].strip()

            if not txt:

                continue

            if line["label"] == "TOC":

                result.append({**line, "page": page})

                added += 1

                continue

            if re.match(r"^\d+(\.\d+)*\.?", txt):

                result.append({**line, "page": page})

                added += 1

                continue

            if txt.upper().startswith("ЗАКЛЮЧЕНИЕ"):

                result.append({**line, "page": page})

                added += 1

                continue

            if txt.upper().startswith("СПИСОК ЛИТЕРАТУРЫ"):

                result.append({**line, "page": page})

                added += 1

                continue

        if added == 0:

            break

        page += 1

    return result

print("\n Применяем фильтр к TOC")
filtered_toc = filter_toc(all_toc_candidates, pages_data)
print(f"Отфильтровано TOC строк: {len(filtered_toc)}")

# фильтруем REFERENCE
def is_ref_heading(text):
    return text.strip().lower() in ["список литературы", "библиографический список", "библиография", "список использованных источников"]

def is_ref_entry_start(text):
    text = text.strip()
    return (re.match(r'^\d+[\.\)]', text) is not None or
            re.match(r'^\[\d+\]', text) is not None or
            re.match(r'^[А-ЯЁ][а-яё]+\s*,\s*[А-ЯЁ]\.', text) is not None or
            re.match(r'^[A-Z][a-z]+\s*,\s*[A-Z]\.', text) is not None)

def filter_reference(candidates, pages_data):
    if not candidates:
        return []

    ref_start_page = None
    for item in candidates:
        if is_ref_heading(item["text"]):
            ref_start_page = item["page"]
            break

    if ref_start_page is None:
        ref_start_page = max(item["page"] for item in candidates)

    result = []
    for item in candidates:
        if item["page"] < ref_start_page - 1:
            continue
        if item["page"] > ref_start_page + 2:
            continue
        result.append(item)

    if not result:
        max_page = max(pages_data.keys())
        for page_num in range(max_page - 2, max_page + 1):
            if page_num not in pages_data:
                continue
            for line in pages_data[page_num]:
                if line["label"] == "REFERENCE" or is_ref_entry_start(line["text"]):
                    result.append({**line, "page": page_num})

    return result

print("\nПрименяем фильтр к REFERENCE")
filtered_ref = filter_reference(all_ref_candidates, pages_data)
print(f"Отфильтровано REFERENCE строк: {len(filtered_ref)}")

# форматирование и сохранение
def format_toc_entry(text):
    match = re.search(r'(.*?)(\d+)$', text.strip())
    if not match:
        return text
    title, page = match.group(1).strip(), match.group(2)
    dots = "." * max(5, 70 - len(title))
    return f"{title} {dots} {page}"

def render_toc(doc, toc_items):
    if not toc_items:
        return

    toc_items = sorted(toc_items, key=lambda x: (x["page"], x["box"][1]))

    p = doc.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    run = p.add_run("СОДЕРЖАНИЕ")
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(14)
    p.paragraph_format.space_after = Pt(12)

    style = doc.styles.add_style('TOCStyle', 1)
    style.font.name = 'Times New Roman'
    style.font.size = Pt(14)
    style.paragraph_format.space_after = Pt(0)
    style.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    tab_stop = Cm(14.5)
    style.paragraph_format.tab_stops.add_tab_stop(tab_stop, WD_TAB_ALIGNMENT.RIGHT, WD_TAB_LEADER.DOTS)

    for item in toc_items:
        if is_toc_heading(item["text"]):
            continue

        text = item["text"].strip()
        match = re.search(r'(.*?)(\d+)$', text)
        if match:
            title = match.group(1).strip()
            page = match.group(2)
            title = re.sub(r'\.{2,}', '', title)
            p = doc.add_paragraph(style=style)
            p.add_run(title + "\t" + page)
        else:
            p = doc.add_paragraph(style=style)
            p.add_run(text)
def split_reference_entries(ref_items):

    ref_items = sorted(
        ref_items,
        key=lambda x: (x["page"], x["box"][1])
    )

    lines = []

    for item in ref_items:

        txt = item["text"].strip()

        if not txt:
            continue

        if is_ref_heading(txt):
            continue

        lines.append(txt)

    entries = []

    current = []

    for line in lines:


        start = False

        if re.match(r"^[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё\-]+[,;]", line):

            start = True

        elif re.match(r"^\d+\.", line):

            start = True

        elif re.match(r"^\[\d+\]", line):

            start = True
        if start:

            if current:

                entries.append(" ".join(current))

            current = [line]

        else:

            if current:

                current.append(line)

            else:

                current = [line]

    if current:

        entries.append(" ".join(current))

    return entries
def render_references(doc, ref_items):

    entries = split_reference_entries(ref_items)

    p = doc.add_paragraph()

    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    r = p.add_run("СПИСОК ЛИТЕРАТУРЫ")

    r.bold = True

    r.font.name = "Times New Roman"

    r.font.size = Pt(14)

    for i, entry in enumerate(entries, 1):

        p = doc.add_paragraph()

        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        p.paragraph_format.first_line_indent = Cm(0)

        run = p.add_run(f"{i}. {entry}")

        run.font.name = "Times New Roman"

        run.font.size = Pt(14)

# сохраняем
doc = Document()
section = doc.sections[0]
section.top_margin = Cm(2)
section.bottom_margin = Cm(2)
section.left_margin = Cm(3)
section.right_margin = Cm(1.5)

render_toc(doc, filtered_toc)
if filtered_toc and filtered_ref:
    doc.add_page_break()
render_references(doc, filtered_ref)

doc.save(OUTPUT_DOCX)
print(f"\nФайл сохранён: {OUTPUT_DOCX}")
print(f"TOC строк {len(filtered_toc)}, REFERENCE строк {len(filtered_ref)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

PDF -> PNG
Извлекаем кандидатов из модели.
Кандидатов TOC: 15
Кандидатов REFERENCE: 70

 Применяем фильтр к TOC
Отфильтровано TOC строк: 15

Применяем фильтр к REFERENCE
Отфильтровано REFERENCE строк: 64

Файл сохранён: /content/drive/MyDrive/GOST_Project/smart_result.docx
TOC строк 15, REFERENCE строк 64
